# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer
from IPython.display import display

df = pd.read_csv("Income Dirty Data.csv")
original_df = df.copy(deep=True)
print(f"Dataset: {df.shape[0]} rows and {df.shape[1]} columns")
display(df.head())

Dataset: 1000 rows and 5 columns


,ID,sex,age,income,tax_15_pct
0,1,Women,21,147168.0,22075.20
1,2,Female,29,119595.0,17939.25
2,3,Female,56,87770.0,13165.50
3,4,NaN,21,54259.0,8138.85
4,5,Male,28,NaN,160230.00


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [2]:
missing_per_column = df.isna().sum()
missing_percentage = df.isna().sum().sum() / df.size * 100
complete_rows = df.notna().all(axis=1).sum()

print("NaN values in each column:")
print(missing_per_column.to_string())
print(f"\nPercentage of all dataset cells that are NaN: {missing_percentage:.2f}%")
print(f"Rows without any NaN values: {complete_rows}")

NaN values in each column:
ID              0
sex            88
age             0
income        109
tax_15_pct     93

Percentage of all dataset cells that are NaN: 5.80%
Rows without any NaN values: 733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [3]:
rule_checks = pd.DataFrame({
    "Age >= 18": df["age"].ge(18),
    "Tax equals 15% of income": (
        df["income"].notna()
        & df["tax_15_pct"].notna()
        & np.isclose(df["tax_15_pct"], 0.15 * df["income"],
                     rtol=0, atol=0.005)
    ),
    "Income > 0": df["income"].gt(0)
})

passes_all = rule_checks.all(axis=1)
print("Percentage of all rows passing each individual rule:")
display((rule_checks.mean() * 100).round(2).rename("Percent passing").to_frame())
print(f"Rows passing all three rules: {passes_all.sum()} of {len(df)}")
print(f"Percentage that violates none of the rules: {passes_all.mean() * 100:.2f}%")

assessable = df[["age", "income", "tax_15_pct"]].notna().all(axis=1)
print(f"Rows with all values needed to check the rules: {assessable.sum()}")
print(f"Passing percentage among those assessable rows: "
      f"{passes_all[assessable].mean() * 100:.2f}%")

Percentage of all rows passing each individual rule:


,Percent passing
Age >= 18,90.2
Tax equals 15% of income,66.2
Income > 0,84.7


Rows passing all three rules: 598 of 1000
Percentage that violates none of the rules: 59.80%
Rows with all values needed to check the rules: 806
Passing percentage among those assessable rows: 74.19%


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [4]:
df["sex"] = df["sex"].replace({
    "Women": "Female", "Woman": "Female",
    "Men": "Male", "Man": "Male"
})
display(df["sex"].value_counts(dropna=False).rename("Count").to_frame())

,Count
sex,
Male,457
Female,455
NaN,88


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [5]:
numeric_columns = ["age", "income", "tax_15_pct"]
for column in numeric_columns:
    non_positive = df[column].le(0)
    print(f"{column}: replaced {non_positive.sum()} non-positive values with NaN")
    df.loc[non_positive, column] = np.nan

print("\nMissing values after these replacements:")
print(df.isna().sum().to_string())

age: replaced 98 non-positive values with NaN
income: replaced 44 non-positive values with NaN
tax_15_pct: replaced 10 non-positive values with NaN

Missing values after these replacements:
ID              0
sex            88
age            98
income        153
tax_15_pct    103


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [6]:
before_imputation = df.copy(deep=True)
example_rows = df.index[df.isna().any(axis=1)][:10]
print("Before imputation:")
display(before_imputation.loc[example_rows])

features = ["sex", "age", "income", "tax_15_pct"]
encoded = df[features].copy()
encoded["sex"] = encoded["sex"].map({"Female": 0, "Male": 1})

scaler = preprocessing.MinMaxScaler()
scaled = scaler.fit_transform(encoded)
imputer = KNNImputer(n_neighbors=5, weights="uniform")
imputed_scaled = imputer.fit_transform(scaled)
imputed_values = pd.DataFrame(
    scaler.inverse_transform(imputed_scaled),
    columns=features, index=df.index
)

completed_df = df.copy(deep=True)
for column in numeric_columns:
    missing = df[column].isna()
    completed_df.loc[missing, column] = imputed_values.loc[missing, column]

missing_sex = df["sex"].isna()
completed_df.loc[missing_sex, "sex"] = np.where(
    imputed_values.loc[missing_sex, "sex"] >= 0.5, "Male", "Female"
)

print("After imputation (same records):")
display(completed_df.loc[example_rows])
print("Remaining missing values:")
print(completed_df.isna().sum().to_string())
assert not completed_df.isna().any().any()
assert completed_df["ID"].equals(original_df["ID"])
assert completed_df["sex"].isin(["Female", "Male"]).all()
for column in features:
    observed = before_imputation[column].notna()
    assert completed_df.loc[observed, column].equals(
        before_imputation.loc[observed, column])
print("\nVerified: all missing values filled; observed values and IDs preserved.")

Before imputation:


,ID,sex,age,income,tax_15_pct
3,4,NaN,21.0,54259.0,8138.85
4,5,Male,28.0,NaN,160230.00
5,6,Female,NaN,128326.0,19248.90
6,7,Female,NaN,NaN,NaN
7,8,Female,24.0,NaN,11820.60
11,12,Male,55.0,NaN,9776.25
13,14,Male,47.0,85300.0,NaN
19,20,Female,NaN,89991.0,NaN
22,23,Male,NaN,NaN,22500.00
27,28,NaN,21.0,76019.0,11402.85


After imputation (same records):


,ID,sex,age,income,tax_15_pct
3,4,Male,21.0,54259.0,8138.85
4,5,Male,28.0,87077.0,160230.00
5,6,Female,36.2,128326.0,19248.90
6,7,Female,38.2,119325.6,16951.50
7,8,Female,24.0,121790.4,11820.60
11,12,Male,55.0,87077.0,9776.25
13,14,Male,47.0,85300.0,13911.30
19,20,Female,27.8,89991.0,16069.05
22,23,Male,34.2,95109.2,22500.00
27,28,Male,21.0,76019.0,11402.85


Remaining missing values:
ID            0
sex           0
age           0
income        0
tax_15_pct    0

Verified: all missing values filled; observed values and IDs preserved.


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

Cleaning a dataset before calculating analytics makes the results more accurate and meaningful. Missing values can reduce the usable sample or bias results if the missing records differ from the observed records. Invalid values, such as negative income or an age of zero, can distort averages and other statistics. Inconsistent labels such as "Woman," "Women," and "Female" can split one group into multiple groups and produce misleading comparisons. Incorrect tax amounts can also lead to inaccurate financial summaries.

Standardizing labels, checking business rules, and handling missing values helps prevent these problems. However, imputed values are estimates rather than known facts, so the method and its assumptions should be documented. Scaling is important for KNN because income and tax are measured on much larger numeric scales than age or encoded sex; without scaling, they would dominate neighbor distances.

The requested replacements in questions 6–8 address only non-positive values. Positive ages below 18 and incorrect positive tax amounts are not automatically corrected by these steps. Likewise, KNN fills missing values but does not guarantee that tax equals exactly 15% of income or that every imputed age is at least 18. Therefore, this completed imputation should not be mistaken for proof that every record satisfies every business rule; further validation and justified corrections would be needed for real analysis.

### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.